In [ ]:

# 2. 전력 품질 피처 (역률 및 비율)
# 무효전력 합계 (전체 흐르는 전기 에너지의 규모)
### [수정] 컬럼명 추후 활용도 높게 수정
# df['Total_Reactive_Power'] = df['Lagging_Current_Reactive_Power_kVarh'] + df['Leading_Current_Reactive_Power_kVarh']

# 인프라가 감당하는 물리적 부하량 (지상+진상 절대 합계)
df['System_Stress_Load'] = df['Lagging_Current_Reactive_Power_kVarh'] + df['Leading_Current_Reactive_Power_kVarh']

# 유효전력 대비 무효전력 비율 (비효율 지표)
# 0으로 나누기 방지를 위해 작은 값(1e-6) 추가
### [수정] 시각화 진행 시 노이즈 방지를 위해 1e-6 추가 처리.
# Usage_kWh가 0이면 0을 부여, 0이 아니면 계산 수행
# 2. 유효전력 대비 무효전력 집약도 (운영 효율 지표)
# Usage가 0인 구간(비가동)은 0으로 처리하여 시각화 스파이크 방지
df['Reactive_Intensity'] = np.where( df['Usage_kWh'] > 0,  df['System_Stress_Load'] / df['Usage_kWh'], 0)


# # 피타고라스 정리를 이용한 종합 역률 계산
# df['Apparent_Power'] = np.sqrt(df['Usage_kWh']**2 + (df['Lagging_Current_Reactive_Power_kVarh'] - df['Leading_Current_Reactive_Power_kVarh'])**2)
# df['Power_Factor'] = df['Usage_kWh'] / (df['Apparent_Power'] + 1e-6)
### [수정] 역률지표의 구분, 1-6승에 대한 부분 처리
# 순무효전력 기반 물리적 역률
df['Apparent_Power'] = np.sqrt(df['Usage_kWh']**2 + (df['Lagging_Current_Reactive_Power_kVarh'] - df['Leading_Current_Reactive_Power_kVarh'])**2)
df['PF_Physical'] = np.where(df['Apparent_Power'] > 0, df['Usage_kWh'] / df['Apparent_Power'], 1.0)


# 3. 변동성 및 추세 피처
# 시계열 데이터이므로 순서대로 정렬되어 있다고 가정
### [수정] 모델을 시계열로 돌리지 않는 상황이므로 전력 사용량에 대한 피쳐는 삭제(다중공선성 우려)
# df['Usage_Moving_Avg_1h_mean'] = df['Usage_kWh'].rolling(window=4).mean() # 1시간(15분*4) 이동평균
# 전력 사용의 흔들림에 대한 내용이므로 유지
df['Usage_Moving_Avg_1h_std'] = df['Usage_kWh'].rolling(window=4).std() # 1시간(15분*4) 이동평균 표준편차


print("파생 피처 생성 완료. 상위 5개 행 확인:")
print(df[['Hour', 'Power_Factor', 'Reactive_Usage_Ratio', 'Production_Capacity_Pct']].head())

In [ ]:
#생산량 가정 피처 
# 연간 최대 전력 사용량을 100% 가동으로 간주
# 데이터 확인 결과 이상치가 없으므로 max를 기준으로 확정
# max_usage = df['Usage_kWh'].max()
# df['Production_Capacity_Pct'] = (df['Usage_kWh'] / max_usage) * 100
### [수정] 생산량 추정에 대해 CO2 피쳐를 활용

# 1. 기기 가동 여부 판별 (Binary)
# CO2 배출이 0보다 크면 가동(1), 아니면 비가동(0)
df['Is_Operating'] = np.where(df['CO2_ppm'] > 0, 1, 0)

# 2. 공정 생산 부하 지표 (Operating Load Index)
# 가동 중일 때의 CO2 수치를 0~1 사이로 스케일링하여 '부하 강도'로 정의
# 비가동 구간(0.00)은 그대로 0으로 둠
max_co2 = df['CO2_ppm'].max()
df['Process_Load_Factor'] = df['CO2_ppm'] / max_co2

# 3. [추가] 분석용 부하 레벨 (Ordinal Category)
# 계단식 데이터를 활용해 '저/중/고' 부하 구간을 명시적으로 구분
# 이는 베이지안 최적화 시 '조건부 최적화'를 위해 매우 유용함
bins = [-np.inf, 0.00, 0.02, 0.05, np.inf]
labels = ['Idle', 'Low_Load', 'Mid_Load', 'High_Load']
df['Load_Level'] = pd.cut(df['CO2_ppm'], bins=bins, labels=labels)

print("CO2 기반 공정 상태 피처 생성이 완료되었습니다.")
print(df[['CO2_ppm', 'Is_Operating', 'Process_Load_Factor', 'Load_Level']].drop_duplicates().sort_values('CO2_ppm'))


In [ ]:

# ---------------------------------
# 1️⃣ 역률 및 무효전력 파생 변수 생성
# ---------------------------------

# 모터 가동 비중 (%) 계산
# 데이터셋 전체에서의 최대 지상 무효전력을 '모터 100% 가동' 상태로 가정
max_lagging = df['Lagging_Current_Reactive_Power_kVarh'].max()
df['Motor_Operating_Rate'] = (df['Lagging_Current_Reactive_Power_kVarh'] / max_lagging) * 100

# 커패시터 가동 비중 (%) 계산
# 최대 진상 무효전력을 '커패시터 100% 가동' 상태로 가정
max_leading = df['Leading_Current_Reactive_Power_kVarh'].max()
df['Capacitor_Operating_Rate'] = (df['Leading_Current_Reactive_Power_kVarh'] / max_leading) * 100

# 무효전력 총량 및 활성비
df['Q_total_abs'] = ( df['Lagging_Current_Reactive_Power_kVarh']
                      + df['Leading_Current_Reactive_Power_kVarh'] )

df['Motor_Ratio'] = ( df['Lagging_Current_Reactive_Power_kVarh'] / df['Q_total_abs'] )

df['Capacitor_Ratio'] = ( df['Leading_Current_Reactive_Power_kVarh'] / df['Q_total_abs'] )


# 콘덴서 과보상 의심 구간
df['Over_Correction_Flag'] = ( (df['Capacitor_Ratio'] > 0.6) & (df['PF_Physical'] > 0.98) )



B. 고주파 변동성 지표 (High-Frequency Volatility)
논리: 앞서 만든 Usage_Moving_Avg_1h_std(표준편차)를 활용합니다. 사용량이 미세하게 진동(Flicker)하거나 급격히 변하는 구간은 고조파가 집중적으로 발생하는 '비정상 조업' 구간일 확률이 큽니다.

피처명: Operational_Instability_Index

3. 고조파 지표 도입 시 모델에 미치는 영향
베이지안 최적화: 고조파 추정치를 **제약 조건(Constraint)**으로 넣으면, "전력량은 줄이되 고조파 스트레스는 높이지 않는" 더 안전한 제어값을 찾을 수 있습니다.

시각화: PF_Physical이 낮은 구간과 고조파 추정치가 높은 구간이 일치하는지 확인하여, 설비 교체가 시급한 '악성 부하'를 특정할 수 있습니다.

3. 피처 보강 전략: "고조파 위험 지수" 정의
Over_Correction_Flag를 버리지 말고, 이를 고조파를 추정하는 핵심 성분으로 활용하여 새로운 지표를 만드는 것이 가장 합리적입니다.

Harmonic_Risk_Index = Over_Correction_Flag + Operational_Instability_Index

논리: 콘덴서가 과보상된 상태(Over_Correction_Flag)에서 사용량의 미세 변동(std)이 크다면, 이를 고조파 발생 가능성이 극도로 높은 **'고위험 구간'**으로 정의합니다.


<table style="border-collapse: collapse; width: 100%; color: #e0e0e0; background-color: #1e1e1e; border: 1px solid #444;">
    <thead>
        <tr style="background-color: #333; color: #ffffff;">
            <th style="padding: 12px; border: 1px solid #444; text-align: left;">분석 계층</th>
            <th style="padding: 12px; border: 1px solid #444; text-align: left;">핵심 피처 (Features)</th>
            <th style="padding: 12px; border: 1px solid #444; text-align: left;">분석적 역할 및 도입 사유</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 10px; border: 1px solid #444; font-weight: bold; color: #81a1c1;">상태 &amp; Context</td>
            <td style="padding: 10px; border: 1px solid #444; font-family: 'Courier New', monospace; color: #a3be8c;">Is_Operating, Process_Load_Factor, Hour, DayOfWeek</td>
            <td style="padding: 10px; border: 1px solid #444;">조업 패턴 및 주기성 학습, CO2 기반 공정 부하 강도 정의</td>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #444; font-weight: bold; color: #81a1c1;">제어 변수 (X)</td>
            <td style="padding: 10px; border: 1px solid #444; font-family: 'Courier New', monospace; color: #a3be8c;">Motor_Operating_Rate, Capacitor_Operating_Rate, Lagging/Leading kVarh</td>
            <td style="padding: 10px; border: 1px solid #444;">직접 제어 가능한 물리적 레버 및 설비별 실제 가동량 소스</td>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #444; font-weight: bold; color: #81a1c1;">성과 &amp; 품질 (Y)</td>
            <td style="padding: 10px; border: 1px solid #444; font-family: 'Courier New', monospace; color: #a3be8c;">PF_Physical, System_Stress_Load, Reactive_Intensity, Usage_kWh</td>
            <td style="padding: 10px; border: 1px solid #444;">최적화 타겟(사용량), 설비 노후화 및 에너지 효율 판단 척도</td>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #444; font-weight: bold; color: #81a1c1;">제약 조건</td>
            <td style="padding: 10px; border: 1px solid #444; font-family: 'Courier New', monospace; color: #a3be8c;">Over_Correction_Flag</td>
            <td style="padding: 10px; border: 1px solid #444;">시스템 안정성 및 고조파 리스크 방지를 위한 가이드라인</td>
        </tr>
    </tbody>
</table>

In [2]:
# ! pip install pypandoc
